# C.1 — Tính tay một đầu attention có mặt nạ nhân quả
**Đề tài 9, tr. 19:** ba token, một đầu chú ý, $d_k=2$. Theo thông tin nhóm cung cấp, giảng viên không phát ma trận riêng. Bài này dùng đúng [bộ minh họa đã chọn](../docs/progress/project-context.md#c1--bộ-ma-trận-minh-họa), không phải số liệu chính thức: Python random.Random(577893521), lấy số nguyên trong $[-2,2]$ theo hàng, lần lượt Q, K, V. Không lấy mẫu lại. Mỗi hàng tương ứng một token; chọn $d_v=2$.

$$Q=\begin{bmatrix}-2&0\\-1&-2\\1&-2\end{bmatrix},\quad K=\begin{bmatrix}-1&-1\\2&1\\0&2\end{bmatrix},\quad V=\begin{bmatrix}-1&0\\-1&1\\0&-1\end{bmatrix}.$$

Tính đầu ra của **head**, chưa qua phép chiếu đầu ra hay residual. Dùng scaled dot-product attention theo [Vaswani et al., §3.2.1–3.2.3](../references/Attention.pdf).

## 1. Nhân Q với K chuyển vị, chia scale rồi che tương lai
Phần tử $(i,j)$ là tích vô hướng của query ở token $i$ và key ở token $j$. Ví dụ, hàng 2 có $(-1)(-1)+(-2)(-1)=3$, $(-1)2+(-2)1=-4$, $(-1)0+(-2)2=-4$. Đặt $c=1/\sqrt{2}$. Ma trận điểm chưa scale và ma trận điểm đã scale, đã che là:

$$QK^\top=\begin{bmatrix}2&-4&0\\3&-4&-4\\1&0&-4\end{bmatrix},\qquad S=cQK^\top+M=\begin{bmatrix}2c&-\infty&-\infty\\3c&-4c&-\infty\\c&0&-4c\end{bmatrix}.$$

Mặt nạ cộng $M_{ij}=0$ nếu $j\le i$, bằng $-\infty$ nếu $j>i$. Che **trước** khi chuẩn hóa từng hàng: $e^{-\infty}=0$, nên token tương lai không góp vào cả tử lẫn mẫu. Đường chéo được giữ.

## 2. Tính softmax từng hàng và tổng có trọng số
Gọi $A=\operatorname{softmax}_{\rm hàng}(S)$ và $o_i=\sum_{j\le i} A_{ij}v_j$. Trừ đi điểm lớn nhất của hàng trước khi lấy mũ không đổi softmax.

- **Token 1:** chỉ có $v_1$ hợp lệ. $A_{1,:}=(1,0,0)$, nên $o_1=v_1=(-1,0)$.
- **Token 2:** trừ $3c$ khỏi hàng 2. Đặt $t=e^{-7c}$, khi đó $A_{2,:}=(1,t,0)/(1+t)$ và $o_2=[(-1,0)+t(-1,1)]/(1+t)=(-1,t/(1+t))$. Tọa độ đầu vẫn là $-1$ vì cả hai value đều có tọa độ đầu bằng $-1$.
- **Token 3:** trừ $c$ khỏi hàng 3. Đặt $u=e^{-c}$, $v=e^{-5c}$ và $D=1+u+v$. Khi đó $A_{3,:}=(1,u,v)/D$ và $o_3=[(-1,0)+u(-1,1)+v(0,-1)]/D=(-(1+u)/D,(u-v)/D)$.

| Token | Trọng số trên các key 1, 2, 3 | Đầu ra trên hai chiều |
|---|---|---|
| 1 | (1, 0, 0) | (−1, 0) |
| 2 | (0.992965, 0.007035, 0) | (−1, 0.007035) |
| 3 | (0.656939, 0.323916, 0.019145) | (−0.980855, 0.304771) |

Bảng làm tròn sau khi tính $O=AV$, không dùng trọng số đã làm tròn để tính tiếp.

## 3. Đối chiếu với mã A.1
Chạy ô dưới từ gốc repo hoặc thư mục notebooks bằng môi trường PyTorch A.1. Đối chiếu tích vô hướng, ma trận trọng số và đầu ra với các biểu thức tính tay ở trên bằng float64 trên CPU; không tải trọng số hay dữ liệu.

In [ ]:
from pathlib import Path
import math
import sys
import torch

here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / 'src/attention.py').is_file()), None)
if root is None:
    raise RuntimeError('Mở notebook từ gốc repo hoặc thư mục notebooks/.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.attention import causal_mask, scaled_dot_product_attention

def matrix(rows):
    return torch.tensor(rows, dtype=torch.float64, device='cpu')

q = matrix([[-2, 0], [-1, -2], [1, -2]])
k = matrix([[-1, -1], [2, 1], [0, 2]])
values = matrix([[-1, 0], [-1, 1], [0, -1]])
torch.testing.assert_close(q @ k.T, matrix([[2, -4, 0], [3, -4, -4], [1, 0, -4]]), rtol=0, atol=0)

c = 1 / math.sqrt(2)
t, u, v = (math.exp(-n*c) for n in (7, 1, 5))
D = 1 + u + v
hand_weights = matrix([[1, 0, 0], [1/(1+t), t/(1+t), 0], [1/D, u/D, v/D]])
hand_output = matrix([[-1, 0], [-1, t/(1+t)], [-(1+u)/D, (u-v)/D]])
output, weights = scaled_dot_product_attention(q, k, values, causal_mask(3, device=q.device))
torch.testing.assert_close(weights, hand_weights, rtol=0, atol=1e-12)
torch.testing.assert_close(output, hand_output, rtol=0, atol=1e-12)
assert weights.triu(1).count_nonzero().item() == 0
for i, row in enumerate(output.tolist(), start=1):
    print(f'Token {i}: ({row[0]:.6f}, {row[1]:.6f})')
print(f'Sai khác đầu ra lớn nhất so với tính tay: {(output - hand_output).abs().max().item():.2e}')